# Deutscher Buchpreis — Metadata Scraper (v2)

Scrapes **author**, **publisher**, and **list membership** (`winner` / `shortlist` / `longlist`) from the official archive at [deutscher-buchpreis.de/archiv](https://www.deutscher-buchpreis.de/archiv).

### How list_type detection works
Each year page has tab sections (Shortlist / Longlist) with thumbnail links referencing `#book-NNN` anchors.  
The scraper collects these IDs to classify each book entry.  
The winner is identified from the "Roman des Jahres" hero block at the top.

In [1]:
# Install dependencies (run once)
# %pip install requests beautifulsoup4 pandas

In [2]:
from dbp_scraper import (
    scrape_year,
    scrape_all_years,
    get_shortlist,
    get_longlist_only,
    merge_metadata,
    scrape_wikipedia_list,
)

## 1. Quick test — single year

In [3]:
entries_2024 = scrape_year(2024)

markers = {"winner": "★", "shortlist": "◆", "longlist": "·"}
for e in entries_2024:
    m = markers[e["list_type"]]
    print(f"{m} {e['title']:45s}  {str(e['author']):22s}  {e['publisher']}")

print(f"\nTotal: {len(entries_2024)} entries")
print("★ = winner, ◆ = shortlist, · = longlist")

· Reichskanzlerplatz                             Nora Bossong            Suhrkamp Verlag
· Seinetwegen                                    Zora del Buono          C. H. Beck
· Die Passagierin                                Franz Friedrich         S. Fischer Verlag
★ Hey guten Morgen, wie geht es dir?             Martina Hefter          Klett-Cotta
· Heilung                                        Timon Karl Kaleyta      Piper Verlag
· Hasenprosa                                     Maren Kames             Suhrkamp Verlag
· Das Philosophenschiff                          Michael Köhlmeier       Carl Hanser Verlag
· Mein drittes Leben                             Daniela Krien           Diogenes Verlag
· Nostalgia                                      André Kubiczek          Rowohlt Berlin
· Das Wohlbefinden                               Ulla Lenze              Klett-Cotta
· Die Projektoren                                Clemens Meyer           S. Fischer Verlag
· Toni & Toni                  

## 2. Full scrape — all years (2005–2025)

One request per year with a 1.5 s delay → ~30 seconds total.

In [4]:
scraped_df = scrape_all_years()
scraped_df.head(10)

Scraping 2005 … OK — 20 entries (1 shortlist/winner)
Scraping 2006 … OK — 21 entries (1 shortlist/winner)
Scraping 2007 … OK — 20 entries (1 shortlist/winner)
Scraping 2008 … OK — 19 entries (1 shortlist/winner)
Scraping 2009 … OK — 20 entries (1 shortlist/winner)
Scraping 2010 … OK — 20 entries (1 shortlist/winner)
Scraping 2011 … OK — 20 entries (1 shortlist/winner)
Scraping 2012 … OK — 20 entries (1 shortlist/winner)
Scraping 2013 … OK — 20 entries (1 shortlist/winner)
Scraping 2014 … OK — 20 entries (1 shortlist/winner)
Scraping 2015 … OK — 20 entries (1 shortlist/winner)
Scraping 2016 … OK — 20 entries (1 shortlist/winner)
Scraping 2017 … OK — 20 entries (1 shortlist/winner)
Scraping 2018 … OK — 20 entries (1 shortlist/winner)
Scraping 2019 … OK — 20 entries (1 shortlist/winner)
Scraping 2020 … OK — 20 entries (1 shortlist/winner)
Scraping 2021 … OK — 20 entries (1 shortlist/winner)
Scraping 2022 … OK — 20 entries (1 shortlist/winner)
Scraping 2023 … OK — 20 entries (1 shortlist/w

,year,title,author,publisher,list_type,book_id
0,2005,Das Geschäftsjahr 1968/69,Bernd Cailloux,Suhrkamp Verlag,longlist,13
1,2005,Spiele,Ulrike Draesner,Luchterhand Literaturverlag,longlist,17
2,2005,Das Fest der Steine,Franzobel,Paul Zsolnay Verlag,longlist,27
3,2005,Es geht uns gut,Arno Geiger,Carl Hanser Verlag,winner,28
4,2005,Die Liebesblödigkeit,Wilhelm Genazino,Carl Hanser Verlag,longlist,29
5,2005,Gezeichnet: Franz Klett,Egon Gramer,Piper Verlag,longlist,37
6,2005,Vanitas oder Hofstätters Begierden,Evelyn Grill,Residenz Verlag,longlist,38
7,2005,Die schwangere Madonna,Peter Henisch,Residenz Verlag,longlist,51
8,2005,Die Vermessung der Welt,Daniel Kehlmann,Rowohlt Verlag,longlist,61
9,2005,42,Thomas Lehr,Aufbau-Verlag,longlist,83


In [19]:
scraped_df = scraped_df.set_index('title')

In [6]:
# Coverage summary
import pandas as pd

summary = (
    scraped_df
    .groupby(["year", "list_type"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["winner", "shortlist", "longlist"], fill_value=0)
)
summary["total"] = summary.sum(axis=1)
print(summary.to_string())
print(f"\nGrand total: {len(scraped_df)} entries")

list_type  winner  shortlist  longlist  total
year                                         
2005            1          0        19     20
2006            1          0        20     21
2007            1          0        19     20
2008            1          0        18     19
2009            1          0        19     20
2010            1          0        19     20
2011            1          0        19     20
2012            1          0        19     20
2013            1          0        19     20
2014            1          0        19     20
2015            1          0        19     20
2016            1          0        19     20
2017            1          0        19     20
2018            1          0        19     20
2019            1          0        19     20
2020            1          0        19     20
2021            1          0        19     20
2022            1          0        19     20
2023            1          0        19     20
2024            1          0      

## 3. Filter: shortlist only

Since your dataset focuses on shortlisted books, here's a quick way to get just those.

In [ ]:
shortlist_df = get_shortlist(scraped_df)
print(f"Shortlisted titles (incl. winners): {len(shortlist_df)}")
shortlist_df[["year", "title", "author", "publisher", "list_type"]].head(15)

## 4. Merge with your existing dataset

Adjust the `pd.read_csv(...)` / `pd.read_json(...)` call to point at your file.  
The merge matches on **title + year** (case-insensitive, whitespace-normalised).

In [7]:
import pandas as pd

# --- Load your existing dataset
df = pd.read_json("deutscher_buchpreis_data.json")

df.head()

,title,author,publisher,year,status,description,jury_rationale
0,Es geht uns gut,TBD,TBD,2005,Winner,"Philipp Erlach hat nie darüber nachgedacht, wa...","Sechzig Jahre österreichische Geschichte, Ansc..."
1,42,TBD,TBD,2005,Shortlist,Als an einem sonnigen Augusttag eine Besucherg...,"Thomas Lehrs ""42"" ist ein Roman, in den ich mi..."
2,Die Vermessung der Welt,TBD,TBD,2005,Shortlist,Ende des 18. Jahrhunderts machen sich zwei jun...,Daniel Kehlmann führt zwei berühmte deutsche N...
3,Dunkle Gesellschaft,TBD,TBD,2005,Shortlist,"Thomas, den Binnenschiffer, hat es von den Flü...","""Dunkle Gesellschaft"" ist ein Roman der Erinne..."
4,So sind wir,TBD,TBD,2005,Shortlist,"Ein Briefbeschwerer, Zeitungen, eine Puppe – i...",Gila Lustiger nähert sich in So sind wir behut...


In [14]:
main_df = df.copy()
main_df = main_df.set_index('title')

In [24]:
for title in main_df.index:
    main_df.loc[title, 'author'] = scraped_df.loc[title, 'author']
    main_df.loc[title, 'publisher'] = scraped_df.loc[title, 'publisher']

In [25]:
main_df

,author,publisher,year,status,description,jury_rationale
title,,,,,,
Es geht uns gut,Arno Geiger,Carl Hanser Verlag,2005,Winner,"Philipp Erlach hat nie darüber nachgedacht, wa...","Sechzig Jahre österreichische Geschichte, Ansc..."
42,Thomas Lehr,Aufbau-Verlag,2005,Shortlist,Als an einem sonnigen Augusttag eine Besucherg...,"Thomas Lehrs ""42"" ist ein Roman, in den ich mi..."
Die Vermessung der Welt,Daniel Kehlmann,Rowohlt Verlag,2005,Shortlist,Ende des 18. Jahrhunderts machen sich zwei jun...,Daniel Kehlmann führt zwei berühmte deutsche N...
Dunkle Gesellschaft,Gert Loschütz,Frankfurter Verlagsanstalt,2005,Shortlist,"Thomas, den Binnenschiffer, hat es von den Flü...","""Dunkle Gesellschaft"" ist ein Roman der Erinne..."
So sind wir,Gila Lustiger,Berlin Verlag,2005,Shortlist,"Ein Briefbeschwerer, Zeitungen, eine Puppe – i...",Gila Lustiger nähert sich in So sind wir behut...
...,...,...,...,...,...,...
Am Samstag gehen die Mädchen in den Wald und jagen Sachen in die Luft,Fiona Sironic,Ecco Verlag,2025,Shortlist,Es brennt. In den Wäldern und auf den Screens....,„Am Samstag gehen die Mädchen in den Wald und ...
Die Ausweichschule,Kaleb Erdmann,park x ullstein,2025,Shortlist,Am letzten Tag der Abiturprüfungen im Jahr 200...,Kaleb Erdmann hat als Elfjähriger den Amoklauf...
Haus zur Sonne,Thomas Melle,Kiepenheuer & Witsch,2025,Shortlist,"Wie viel Selbstbestimmung ist möglich, wenn da...","Thomas Melle erzählt von extremen Höhenflügen,..."


## 6. Save enriched dataset

In [27]:
main_df.to_csv("deutscher_buchpreis_enriched.csv", index=True)
print("Saved to deutscher_buchpreis_enriched.csv ✓")

Saved to deutscher_buchpreis_enriched.csv ✓


In [31]:
sample_df = pd.read_csv('deutscher_buchpreis_enriched.csv')

In [35]:
sample_df[:2].to_csv('sample_deutscher_buchpreis.csv', index=False)